# Zilliz vector database

In [ ]:
import json
import os
from pathlib import Path
from pymilvus import MilvusClient, DataType
from dotenv import load_dotenv

In [ ]:
BASE_DIR = Path(r"D:\\Final_GRAG")
load_dotenv(BASE_DIR / ".env")

ZILLIZ_CLOUD_URI = os.getenv("ZILLIZ_CLOUD_URI")
ZILLIZ_CLOUD_API_KEY = os.getenv("ZILLIZ_CLOUD_API_KEY")

UNITS_COLLECTION_NAME = "gri_units_with_sector_metadata"
UNITS_JSON_PATH = BASE_DIR / "metadata" / "gri_units" / "gri_units_with_sector_metadata.json"

In [ ]:
# Tạo kết nối tới Zilliz Cloud bằng pymilvus
client = MilvusClient(
    uri=ZILLIZ_CLOUD_URI,
    token=ZILLIZ_CLOUD_API_KEY
)

## Collection: gri_units_with_sector_metadata

In [ ]:
gri_units_schema = MilvusClient.create_schema(
    auto_id=False,
    enable_dynamic_field=True,
)

gri_units_schema.add_field(
    field_name="unit_id",
    datatype=DataType.VARCHAR,
    max_length=64,
    is_primary=True
)

gri_units_schema.add_field(
    field_name="dense_embedding",
    datatype=DataType.FLOAT_VECTOR,
    dim=1024
)

gri_units_schema.add_field(
    field_name="sparse_embedding",
    datatype=DataType.SPARSE_FLOAT_VECTOR
)

gri_units_schema.add_field(
    field_name="standard_id",
    datatype=DataType.VARCHAR,
    max_length=32
)

gri_units_schema.add_field(
    field_name="standard_name",
    datatype=DataType.VARCHAR,
    max_length=256
)

gri_units_schema.add_field(
    field_name="standard_type",
    datatype=DataType.VARCHAR,
    max_length=32
)

gri_units_schema.add_field(
    field_name="disclosure_id",
    datatype=DataType.VARCHAR,
    max_length=32
)

gri_units_schema.add_field(
    field_name="disclosure_name",
    datatype=DataType.VARCHAR,
    max_length=512
)

gri_units_schema.add_field(
    field_name="requirement_id",
    datatype=DataType.VARCHAR,
    max_length=32
)

gri_units_schema.add_field(
    field_name="requirement_text",
    datatype=DataType.VARCHAR,
    max_length=8192
)

gri_units_schema.add_field(
    field_name="requirement_type",
    datatype=DataType.VARCHAR,
    max_length=32
)

gri_units_schema.add_field(
    field_name="is_omittable",
    datatype=DataType.BOOL
)

gri_units_schema.add_field(
    field_name="year",
    datatype=DataType.INT16
)

gri_units_schema.add_field(
    field_name="hierarchy_level",
    datatype=DataType.INT8
)

gri_units_schema.add_field(
    field_name="parent_requirement",
    datatype=DataType.VARCHAR,
    max_length=32
)

gri_units_schema.add_field(
    field_name="standards_id",
    datatype=DataType.VARCHAR,
    max_length=128
)

gri_units_schema.add_field(
    field_name="sector_name",
    datatype=DataType.VARCHAR,
    max_length=256
)

gri_units_schema.add_field(
    field_name="topic_id",
    datatype=DataType.VARCHAR,
    max_length=256
)

gri_units_schema.add_field(
    field_name="topic_name",
    datatype=DataType.VARCHAR,
    max_length=1024
)

In [ ]:
index_params = client.prepare_index_params()

index_params.add_index(field_name="unit_id")

# Index cho Dense vector (semantic/cosine similarity search)
index_params.add_index(
    field_name="dense_embedding",
    index_type="AUTOINDEX",
    metric_type="COSINE"
)

# Index cho Sparse vector
index_params.add_index(
    field_name="sparse_embedding",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="IP"
)

index_params.add_index(field_name="standard_id")
index_params.add_index(field_name="standard_type")
index_params.add_index(field_name="disclosure_id")
index_params.add_index(field_name="is_omittable")
index_params.add_index(field_name="standards_id")
index_params.add_index(field_name="sector_name")
index_params.add_index(field_name="topic_id")

In [ ]:
if UNITS_COLLECTION_NAME in client.list_collections():
    client.drop_collection(UNITS_COLLECTION_NAME)

# Tạo collection units mới
client.create_collection(
    collection_name=UNITS_COLLECTION_NAME,
    schema=gri_units_schema,
    index_params=index_params
)

In [ ]:
with open(UNITS_JSON_PATH, 'r', encoding='utf-8') as f:
    units_data = json.load(f)

print(len(units_data))

In [ ]:
FIELD_MAX_LENGTHS = {
    'standard_id': 32,
    'standard_name': 256,
    'standard_type': 32,
    'disclosure_id': 32,
    'disclosure_name': 512,
    'requirement_id': 32,
    'requirement_text': 8192,
    'requirement_type': 32,
    'parent_requirement': 32,
    'standards_id': 128,
    'sector_name': 256,
    'topic_id': 256,
    'topic_name': 1024,
}

prepared_units = []
for unit in units_data:
    prepared_unit = unit.copy()

    # Backward-compat: dữ liệu cũ có thể dùng is_mandatory
    if 'is_omittable' not in prepared_unit and 'is_mandatory' in prepared_unit:
        prepared_unit['is_omittable'] = prepared_unit.pop('is_mandatory')

    # Drop các field legacy không còn cần dùng
    prepared_unit.pop('claim_level', None)
    prepared_unit.pop('sector_applicability', None)

    if 'sparse_embedding' in prepared_unit:
        sparse = prepared_unit['sparse_embedding']
        if isinstance(sparse, dict) and 'indices' in sparse and 'values' in sparse:
            prepared_unit['sparse_embedding'] = dict(zip(sparse['indices'], sparse['values']))
        elif not isinstance(sparse, dict):
            print(f"Sai định dạng sparse embedding ở unit {prepared_unit.get('unit_id')}")

    # Chuẩn hóa null + ép kiểu + cắt chuỗi theo schema VARCHAR
    for text_field, max_len in FIELD_MAX_LENGTHS.items():
        value = prepared_unit.get(text_field)
        if value is None:
            prepared_unit[text_field] = ""
        else:
            prepared_unit[text_field] = str(value)[:max_len]

    if prepared_unit.get('is_omittable') is None:
        prepared_unit['is_omittable'] = False

    prepared_units.append(prepared_unit)

print(len(prepared_units))

In [ ]:
batch_size = 100

for i in range(0, len(prepared_units), batch_size):
    batch = prepared_units[i:i+batch_size]
    result = client.insert(
        collection_name=UNITS_COLLECTION_NAME,
        data=batch
    )